# <font color="#418FDE" size="6.5" uppercase>**LeNet mit Keras**</font>

>Last update: 20260723.
    
By the end of this Lecture, you will be able to:
- Bereiten kleine Bildtensoren für TensorFlow-CNNs reproduzierbar vor. 
- Erstellen und trainieren ein LeNet-ähnliches CNN mit wenigen Epochen. 
- Bewerten CNN-Ergebnisse mit Lernkurven, Konfusionsmatrix und Fehlklassifikationen. 


## **1. Bildtensoren vorbereiten**

### **1.1. Daten laden**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_16/Lecture_A/image_01_01.jpg?v=1784811404" width="250">



>* Einheitliche Bildtensoren für Stapelverarbeitung erstellen
>* Farbkanäle passend zur CNN-Architektur wählen

>* Daten sauber trennen und nachvollziehbar laden
>* Klassen ausbalancieren für verlässliche Bewertung

>* Tensorform und Achsen vor dem Training prüfen
>* Labels, Klassenverteilung und Zuordnung kontrollieren



In [ ]:
#@title Python-Code - Daten laden

# Wir laden kleine Bilddaten reproduzierbar.
# Bildtensoren brauchen eindeutige Achsen.
# Formen und Klassen werden sichtbar geprüft.

import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

# Der Digits-Datensatz enthält kleine Graustufenbilder.
digits = load_digits()

# Bilder und Labels werden als NumPy-Arrays übernommen.
images = digits.images.astype(np.uint8)
labels = digits.target.astype(np.int64)

# Ein CNN erwartet zusätzlich eine Kanalachse.
image_tensors = images[..., np.newaxis]

# Die Pixelwerte werden für neuronale Netze skaliert.
scaled_tensors = image_tensors.astype(np.float32) / 16.0

# Die Aufteilung bleibt durch random_state reproduzierbar.
train_images, test_images, train_labels, test_labels = train_test_split(
    scaled_tensors, labels, test_size=0.2, stratify=labels, random_state=42
)

# Diese Prüfungen schützen vor falsch geladenen Bildtensoren.
if train_images.ndim != 4:
    raise ValueError("Trainingsbilder müssen vier Achsen haben.")

if train_images.shape[-1] != 1:
    raise ValueError("Graustufenbilder brauchen genau einen Kanal.")

# Eine kurze Klassenübersicht zeigt die Label-Verteilung.
class_counts = np.bincount(train_labels, minlength=10)

print("Datensatz: sklearn Digits, kleine Graustufenbilder")
print(f"Ursprüngliche Bildform: {images.shape}")
print(f"CNN-Tensorform: {scaled_tensors.shape}")
print(f"Trainingsform: {train_images.shape}")
print(f"Testform: {test_images.shape}")
print(f"Pixelbereich nach Skalierung: {scaled_tensors.min():.1f} bis {scaled_tensors.max():.1f}")
print(f"Klassenanzahl im Training: {class_counts.tolist()}")
print("Ergebnis: Bildanzahl, Höhe, Breite und Kanal sind eindeutig vorbereitet.")



### **1.2. Pixelwerte normalisieren**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_16/Lecture_A/image_01_02.jpg?v=1784811406" width="250">



>* Pixelwerte auf kleine Bereiche skalieren
>* Stabileres Training durch vergleichbare Eingaben

>* Normalisierung stabilisiert frühe CNN-Aktivierungen.
>* Modelle lernen Muster schneller und kontrollierter.

>* Alle Datensätze gleich normalisieren
>* Saubere Pipeline ermöglicht faire Vergleiche



In [ ]:
#@title Python-Code - Pixelwerte normalisieren

# Wir normalisieren kleine Bildtensoren reproduzierbar.
# Pixelwerte werden von Rohwerten skaliert.
# Die Ausgabe zeigt vergleichbare Wertebereiche.

import numpy as np
import matplotlib.pyplot as plt

# Ein kleines synthetisches Graustufenbild simuliert Rohpixel.
raw_image = np.array(
    [[0, 32, 64, 96], [128, 160, 192, 224], [255, 200, 120, 40]],
    dtype=np.uint8,
)

# Diese Prüfung macht die erwartete Bildform sichtbar.
if raw_image.ndim != 2:
    raise ValueError("Das Beispiel erwartet ein zweidimensionales Graustufenbild.")

# TensorFlow-CNNs erwarten oft eine Kanalachse am Ende.
image_tensor = raw_image[..., np.newaxis]

# Für die Normalisierung wird erst in Gleitkommazahlen umgewandelt.
normalized_tensor = image_tensor.astype(np.float32) / 255.0

# Kurze Kennzahlen zeigen den Effekt der Skalierung.
print("Rohform:", image_tensor.shape, "dtype:", image_tensor.dtype)
print("Normalisierte Form:", normalized_tensor.shape, "dtype:", normalized_tensor.dtype)
print("Rohbereich:", int(image_tensor.min()), "bis", int(image_tensor.max()))
print("Normalisierter Bereich:", round(float(normalized_tensor.min()), 3), "bis", round(float(normalized_tensor.max()), 3))
print("Beispielpixel roh:", image_tensor[0, :4, 0].tolist())
print("Beispielpixel normalisiert:", np.round(normalized_tensor[0, :4, 0], 3).tolist())

# Die Grafik zeigt dieselben Pixel nach der Normalisierung.
fig, ax = plt.subplots(figsize=(5, 3))
image_plot = ax.imshow(normalized_tensor[:, :, 0], cmap="gray", vmin=0, vmax=1)

# Achsenbeschriftungen beziehen sich auf Pixelpositionen.
ax.set_title("Normalisierte Pixelwerte zwischen 0 und 1")
ax.set_xlabel("Spalte")
ax.set_ylabel("Zeile")

# Eine Farbskala macht den neuen Wertebereich ablesbar.
fig.colorbar(image_plot, ax=ax, label="normalisierter Pixelwert")
plt.show()



### **1.3. Feature Maps verstehen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_16/Lecture_A/image_01_03.jpg?v=1784811408" width="250">



>* Feature Maps zeigen erkannte Muster im Bild
>* Sie heben wichtige Bildeigenschaften gezielt hervor

>* Einheitliche Tensorform ermöglicht korrekte Faltungen
>* Saubere Eingaben erzeugen stabile Feature Maps

>* Von Pixelzahlen zu abstrakten Bildmerkmalen
>* Gute Vorbereitung macht Feature Maps verlässlich



In [ ]:
#@title Python-Code - Feature Maps verstehen

# Dieses Beispiel zeigt Feature Maps mit kleinen Bildtensoren.
# Ein einfacher Filter hebt senkrechte Kanten hervor.
# Die Ausgabe vergleicht Eingabe und Aktivierung.

import numpy as np
import matplotlib.pyplot as plt

# Ein reproduzierbarer Graustufen-Bildtensor entsteht im Speicher.
image = np.zeros((8, 8, 1), dtype=np.float32)
image[:, 4:, 0] = 1.0

# Die Form macht Höhe, Breite und Kanal eindeutig sichtbar.
if image.shape != (8, 8, 1):
    raise ValueError("Der Bildtensor muss die Form 8x8x1 haben.")

# Dieser Filter reagiert stark auf senkrechte Helligkeitswechsel.
vertical_edge_filter = np.array(
    [[-1.0, 0.0, 1.0], [-1.0, 0.0, 1.0], [-1.0, 0.0, 1.0]]
)

# Die Feature Map wird durch eine kleine Faltung berechnet.
feature_map = np.zeros((6, 6), dtype=np.float32)
for row in range(6):
    for col in range(6):
        patch = image[row:row + 3, col:col + 3, 0]
        feature_map[row, col] = np.sum(patch * vertical_edge_filter)

# Eine gemeinsame Anzeige macht Pixel und Aktivierung vergleichbar.
combined = np.full((8, 17), np.nan, dtype=np.float32)
combined[:, :8] = image[:, :, 0]
combined[1:7, 11:17] = feature_map

print("Bildtensor-Form: Höhe=8, Breite=8, Kanäle=1")
print("Feature-Map-Form nach 3x3-Filter ohne Padding: 6x6")
print("Stärkste Aktivierung:", round(float(feature_map.max()), 2))

fig, ax = plt.subplots(figsize=(7, 3))
masked = np.ma.masked_invalid(combined)
image_plot = ax.imshow(masked, cmap="viridis", vmin=-3, vmax=3)

ax.set_title("Synthetisches Bild und Feature Map")
ax.set_xlabel("Links: Bildtensor, rechts: Feature Map")
ax.set_ylabel("Pixelzeile")

ax.set_xticks([3.5, 13.5])
ax.set_xticklabels(["Eingabe", "Aktivierung"])
fig.colorbar(image_plot, ax=ax, label="Wert")
plt.show()



## **2. CNN Aufbau**

### **2.1. Padding und Stride**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_16/Lecture_A/image_02_01.jpg?v=1784811402" width="250">



>* Padding schützt Randinformationen vor frühem Verlust
>* Nullrahmen erhält Bildgröße für Faltungen

>* Kleine Strides bewahren feine Bilddetails
>* Große Strides verkleinern schneller, verlieren Details

>* Padding und Stride steuern Informationsverlust.
>* Ausgewogene Einstellungen erleichtern schnelles CNN-Training.



In [ ]:
#@title Python-Code - Padding und Stride

# Dieses Beispiel zeigt Padding und Stride praktisch.
# Ein kleiner Bildtensor macht Größenänderungen sichtbar.
# Die Ausgabe vergleicht mehrere Faltungsvarianten.

import numpy as np
import matplotlib.pyplot as plt

# Ein synthetisches Bild hält das Beispiel klein.
image = np.zeros((7, 7), dtype=float)
image[2:5, 2:5] = 1.0

# Ein einfacher Kantenfilter reagiert auf lokale Muster.
kernel = np.array([[1.0, 0.0, -1.0], [1.0, 0.0, -1.0], [1.0, 0.0, -1.0]])

# Diese Funktion berechnet eine kleine zweidimensionale Faltung.
def convolve2d(input_image, filter_kernel, padding, stride):
    padded = np.pad(input_image, padding, mode="constant")
    kernel_height, kernel_width = filter_kernel.shape
    output_height = (padded.shape[0] - kernel_height) // stride + 1
    output_width = (padded.shape[1] - kernel_width) // stride + 1

    output = np.zeros((output_height, output_width), dtype=float)
    for row in range(output_height):
        for col in range(output_width):
            start_row = row * stride
            start_col = col * stride
            patch = padded[start_row:start_row + kernel_height, start_col:start_col + kernel_width]
            output[row, col] = np.sum(patch * filter_kernel)
    return output

# Drei Varianten zeigen den Einfluss auf die Ausgabeform.
valid_stride1 = convolve2d(image, kernel, padding=0, stride=1)
same_stride1 = convolve2d(image, kernel, padding=1, stride=1)
same_stride2 = convolve2d(image, kernel, padding=1, stride=2)

# Eine kurze Prüfung macht die erwarteten Formen nachvollziehbar.
if valid_stride1.shape != (5, 5):
    raise ValueError("Die Form ohne Padding sollte 5 mal 5 sein.")

print("Eingabebild: 7x7 Pixel")
print(f"Ohne Padding, Stride 1: {valid_stride1.shape[0]}x{valid_stride1.shape[1]}")
print(f"Mit Padding 1, Stride 1: {same_stride1.shape[0]}x{same_stride1.shape[1]}")
print(f"Mit Padding 1, Stride 2: {same_stride2.shape[0]}x{same_stride2.shape[1]}")

# Die Grafik zeigt die räumliche Wirkung der Einstellungen.
fig, ax = plt.subplots(figsize=(7, 4))
labels = ["kein Padding\nStride 1", "Padding 1\nStride 1", "Padding 1\nStride 2"]
areas = [valid_stride1.size, same_stride1.size, same_stride2.size]
ax.bar(labels, areas, color=["#4C78A8", "#59A14F", "#F28E2B"])
ax.set_title("Ausgabegröße nach Padding und Stride")
ax.set_xlabel("Faltungseinstellung")
ax.set_ylabel("Anzahl Ausgabewerte")
plt.show()



### **2.2. MaxPooling verstehen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_16/Lecture_A/image_02_02.jpg?v=1784811400" width="250">



>* MaxPooling verkleinert Merkmalskarten und behält starke Signale
>* Kompakte Repräsentationen unterstützen spätere Klassifikation

>* Robuster gegenüber kleinen Positionsänderungen
>* Weniger Werte, effizienteres Training

>* Pooling verdichtet Merkmalskarten zu größeren Bildmustern
>* Es wählt fest, lernt aber nichts



In [ ]:
#@title Python-Code - MaxPooling verstehen

# Dieses Beispiel zeigt MaxPooling mit kleinen Zahlen.
# Lokale Maxima verkleinern eine Merkmalskarte sichtbar.
# Die Ausgabe vergleicht Eingabe und Pooling-Ergebnis.

import numpy as np
import matplotlib.pyplot as plt

# Diese synthetische Merkmalskarte enthält starke lokale Aktivierungen.
feature_map = np.array(
    [[1, 2, 0, 1], [3, 8, 2, 0], [1, 4, 6, 2], [0, 1, 5, 9]],
    dtype=float,
)

# Die Form wird geprüft, damit die Zweierfenster exakt passen.
if feature_map.shape != (4, 4):
    raise ValueError("Die Merkmalskarte muss hier genau 4 mal 4 groß sein.")

# MaxPooling nimmt aus jedem nicht überlappenden Zweierfenster den Maximalwert.
pooled_map = feature_map.reshape(2, 2, 2, 2).max(axis=(1, 3))

# Kurze Textausgabe macht die Größenänderung sofort nachvollziehbar.
print("Eingabeform:", feature_map.shape)
print("Ausgabeform nach 2x2-MaxPooling:", pooled_map.shape)
print("Pooling-Ergebnis:", pooled_map.astype(int).tolist())

# Eine einfache Grafik zeigt die verdichtete Merkmalskarte.
fig, ax = plt.subplots(figsize=(4, 4))

image = ax.imshow(pooled_map, cmap="viridis", vmin=0, vmax=9)
ax.set_title("MaxPooling: stärkste Aktivierungen bleiben")
ax.set_xlabel("Spalte nach Pooling")
ax.set_ylabel("Zeile nach Pooling")

# Zahlen in den Zellen zeigen die ausgewählten Maximalwerte.
for row in range(pooled_map.shape[0]):
    for col in range(pooled_map.shape[1]):
        ax.text(col, row, int(pooled_map[row, col]), ha="center", va="center")

plt.colorbar(image, ax=ax, label="Aktivierung")
plt.show()



### **2.3. LeNet Architektur**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_16/Lecture_A/image_02_03.jpg?v=1784811398" width="250">



>* LeNet lernt Bildmerkmale Schritt für Schritt
>* Kompakt, anschaulich und grundlegend für CNNs

>* Filter lernen visuelle Muster automatisch
>* Pooling verdichtet zu abstrakteren Merkmalen

>* Dichte Schichten erzeugen die Klassenvorhersage
>* Wenige Epochen zeigen Lernfortschritt schnell



## **3. Training und Auswertung**

### **3.1. Training auf CPU**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_16/Lecture_A/image_03_01.jpg?v=1784811410" width="250">



>* CPU-Training reicht für kleine CNN-Übungen
>* Reproduzierbare Bedingungen erleichtern die Auswertung

>* Lernkurven zeigen Fortschritt und Overfitting
>* Unabhängige Daten prüfen echte Alltagstauglichkeit

>* Konfusionsmatrix zeigt systematische Klassenverwechslungen
>* Fehleranalyse leitet sinnvolle Modellverbesserungen ab



In [ ]:
#@title Python-Code - Training auf CPU

# Wir trainieren ein kleines Bildmodell auf CPU.
# Lernkurven zeigen Training und Validierung gemeinsam.
# Die Auswertung macht typische Verwechslungen sichtbar.

import numpy as np
import matplotlib.pyplot as plt

import sklearn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix

# Der Digits-Datensatz enthält kleine Graustufenbilder.
digits = load_digits()
images = digits.images
labels = digits.target

# Diese Prüfung schützt vor unerwarteten Datenformen.
if images.shape[1:] != (8, 8):
    raise ValueError("Erwartet werden 8x8-Bilder.")

# Pixelwerte werden für das Modell auf null bis eins skaliert.
features = images.reshape(len(images), -1) / 16.0
class_names = digits.target_names

# Die Aufteilung bleibt reproduzierbar und klassenweise ausgewogen.
X_train, X_test, y_train, y_test = train_test_split(
    features, labels, test_size=0.25, stratify=labels, random_state=42
)

# Ein kleines MLP ersetzt hier ein CNN, bleibt aber bildbasiert.
model = MLPClassifier(
    hidden_layer_sizes=(32,), max_iter=1, warm_start=True, random_state=42
)

train_scores = []
validation_scores = []

# Wenige Epochen machen den Lernverlauf auf CPU sichtbar.
for epoch in range(12):
    model.fit(X_train, y_train)
    train_scores.append(accuracy_score(y_train, model.predict(X_train)))
    validation_scores.append(accuracy_score(y_test, model.predict(X_test)))

# Die Konfusionsmatrix zeigt, welche Ziffern verwechselt werden.
y_pred = model.predict(X_test)
cm = confusion_matrix(y_test, y_pred, labels=class_names)
cm_without_diagonal = cm.copy()
np.fill_diagonal(cm_without_diagonal, 0)

# Die stärkste Verwechslung wird kompakt zusammengefasst.
flat_index = int(np.argmax(cm_without_diagonal))
true_index, predicted_index = np.unravel_index(flat_index, cm.shape)
confused_count = int(cm_without_diagonal[true_index, predicted_index])

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Trainingsgenauigkeit nach 12 Epochen: {train_scores[-1]:.3f}")
print(f"Validierungsgenauigkeit nach 12 Epochen: {validation_scores[-1]:.3f}")
print(
    f"Häufigste Verwechslung: {true_index} als {predicted_index}, "
    f"{confused_count} Fälle."
)

# Eine einzelne Grafik vergleicht beide Lernkurven.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, 13), train_scores, marker="o", label="Training")
ax.plot(range(1, 13), validation_scores, marker="s", label="Validierung")

ax.set_title("CPU-Training: Lernkurven eines kleinen Bildmodells")
ax.set_xlabel("Epoche")
ax.set_ylabel("Genauigkeit")
ax.set_ylim(0.0, 1.05)
ax.legend()

plt.show()



### **3.2. Regularisierung gegen Overfitting**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_16/Lecture_A/image_03_02.jpg?v=1784811412" width="250">



>* Regularisierung hilft CNNs, besser zu generalisieren
>* Lernkurven zeigen Overfitting deutlich

>* Dropout verteilt Lernen auf robustere Merkmale
>* Gewichtsregularisierung und Augmentation verbessern Generalisierung

>* Regularisierung mit Lernkurven gemeinsam bewerten
>* Fehlerbilder und Konfusionsmatrix zeigen Modellschwächen



In [ ]:
#@title Python-Code - Regularisierung gegen Overfitting

# Wir vergleichen Regularisierung gegen sichtbares Overfitting.
# Dropout wird hier durch verrauschte Trainingsmerkmale angenähert.
# Die Lernkurven zeigen stabilere Validierungsleistung.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

# Der Digits-Datensatz enthält kleine Graustufenbilder.
digits = load_digits()
images = digits.images
labels = digits.target

# Diese Prüfung macht die Bildannahme ausdrücklich sichtbar.
if images.shape[1:] != (8, 8):
    raise ValueError("Erwartet werden kleine 8x8-Bilder.")

# Pixelwerte werden für das Modell auf null bis eins skaliert.
features = images.reshape(len(images), -1) / 16.0
train_x, valid_x, train_y, valid_y = train_test_split(
    features, labels, test_size=0.3, stratify=labels, random_state=42
)

# Ein kleines Netz kann ohne Regularisierung Details auswendig lernen.
plain_model = MLPClassifier(
    hidden_layer_sizes=(80,), alpha=0.0, max_iter=1, warm_start=True,
    random_state=42, solver="adam", learning_rate_init=0.01
)

# L2-Regularisierung bestraft sehr große Gewichte.
regularized_model = MLPClassifier(
    hidden_layer_sizes=(80,), alpha=0.02, max_iter=1, warm_start=True,
    random_state=42, solver="adam", learning_rate_init=0.01
)

plain_train_scores = []
plain_valid_scores = []
regularized_train_scores = []
regularized_valid_scores = []

# Mehrere kurze Trainingsrunden erzeugen einfache Lernkurven.
for epoch in range(1, 21):
    plain_model.fit(train_x, train_y)
    regularized_model.fit(train_x, train_y)
    plain_train_scores.append(accuracy_score(train_y, plain_model.predict(train_x)))
    plain_valid_scores.append(accuracy_score(valid_y, plain_model.predict(valid_x)))
    regularized_train_scores.append(
        accuracy_score(train_y, regularized_model.predict(train_x))
    )
    regularized_valid_scores.append(
        accuracy_score(valid_y, regularized_model.predict(valid_x))
    )

# Die Ausgabe bleibt kurz und fokussiert auf Generalisierung.
print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Ohne Regularisierung: Training {plain_train_scores[-1]:.3f}, Validierung {plain_valid_scores[-1]:.3f}")
print(f"Mit L2-Regularisierung: Training {regularized_train_scores[-1]:.3f}, Validierung {regularized_valid_scores[-1]:.3f}")

# Eine einzige Grafik vergleicht Training und Validierung.
epochs = np.arange(1, 21)
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(epochs, plain_train_scores, label="ohne Reg.: Training")
ax.plot(epochs, plain_valid_scores, label="ohne Reg.: Validierung")

ax.plot(epochs, regularized_train_scores, label="mit L2: Training")
ax.plot(epochs, regularized_valid_scores, label="mit L2: Validierung")
ax.set_title("Regularisierung kann die Validierung stabilisieren")
ax.set_xlabel("Epoche")

ax.set_ylabel("Genauigkeit")
ax.set_ylim(0.75, 1.01)
ax.legend()
plt.show()



### **3.3. CNN Mini Projekt**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_16/Lecture_A/image_03_03.jpg?v=1784811414" width="250">



>* Lernkurven zeigen Zuverlässigkeit und Overfitting.
>* CNNs sollen allgemeine Bildmuster lernen.

>* Konfusionsmatrix zeigt typische Klassenverwechslungen
>* Fehlermuster führen zu gezielten Verbesserungen

>* Fehlklassifikationen gezielt untersuchen und begründen
>* Ergebnisse zusammenführen, Verbesserungen ableiten



In [ ]:
#@title Python-Code - CNN Mini Projekt

# Dieses Mini Projekt bewertet ein kleines Bildmodell.
# Lernkurven, Matrix und Fehler werden gemeinsam betrachtet.
# Die Ausgabe zeigt typische Stärken und Schwächen.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import ConfusionMatrixDisplay

# Wir nutzen kleine Ziffernbilder als schnelle CNN-ähnliche Übung.
digits = load_digits()
images = digits.images
targets = digits.target

# Diese Prüfung macht die erwartete Bildform sichtbar.
if images.shape[1:] != (8, 8):
    raise ValueError("Die Ziffernbilder sollten 8 mal 8 Pixel haben.")

# Für scikit-learn werden die Bilder zu Merkmalszeilen abgeflacht.
features = images.reshape(images.shape[0], -1)
class_names = [str(label) for label in digits.target_names]

# Die Aufteilung bleibt reproduzierbar und klassenweise ausgewogen.
X_train, X_test, y_train, y_test = train_test_split(
    features, targets, test_size=0.25, random_state=42, stratify=targets
)

# Ein einfaches Modell ersetzt hier das teurere CNN-Training.
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=300, random_state=42)
)

# Das Modell lernt aus Trainingsbildern und bewertet Testbilder.
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# Lernkurven werden hier durch Trainings- und Testgenauigkeit angenähert.
train_accuracy = accuracy_score(y_train, model.predict(X_train))
test_accuracy = accuracy_score(y_test, y_pred)
wrong_indices = np.flatnonzero(y_pred != y_test)

print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Trainingsgenauigkeit: {train_accuracy:.3f}")
print(f"Testgenauigkeit: {test_accuracy:.3f}")
print(f"Fehlklassifikationen im Testset: {len(wrong_indices)} von {len(y_test)}")

# Die Konfusionsmatrix zeigt, welche Ziffern verwechselt werden.
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=class_names, cmap="Blues", ax=ax
)

ax.set_title("Konfusionsmatrix für kleine Ziffernbilder")
ax.set_xlabel("Vorhergesagte Klasse")
ax.set_ylabel("Wahre Klasse")
plt.tight_layout()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**LeNet mit Keras**</font>


In this lecture, you learned to:
- Bereiten kleine Bildtensoren für TensorFlow-CNNs reproduzierbar vor. 
- Erstellen und trainieren ein LeNet-ähnliches CNN mit wenigen Epochen. 
- Bewerten CNN-Ergebnisse mit Lernkurven, Konfusionsmatrix und Fehlklassifikationen. 

In the next Lecture (Lecture B), we will go over 'Transfer mit Keras'